In [1]:
import numpy as np
import pandas as pd

file_path = 'other_side.csv'

bursts = pd.read_csv(file_path)

print(bursts.head())

        Start time         End time  High frequency (Hz)  Low frequency (Hz)  \
0   2023-2-8/03:20   2023-2-8/03:40              6000000              200000   
1   2023-2-8/04:05   2023-2-8/04:25              2000000              400000   
2   2023-2-9/07:15   2023-2-9/07:30             20000000              300000   
3  2023-2-10/02:56  2023-2-10/03:05             20000000              500000   
4  2023-2-11/00:50  2023-2-11/01:05             10000000              600000   

  GOES Flare Notes (corona loop brightening, etc.)    Other notes  
0    F   NaN                                   NaN            NaN  
1    F   NaN                                   NaN            NaN  
2    T     F                                   NaN  SDO not shown  
3    F   NaN                                   NaN            NaN  
4    F   NaN                                   NaN            NaN  


In [3]:
freq_change = bursts["High frequency (Hz)"] - bursts["Low frequency (Hz)"]
bursts["Frequency Change"] = freq_change

bursts['Start time'] = pd.to_datetime(bursts['Start time'], format='%Y-%m-%d/%H:%M', errors='coerce')
bursts['End time'] = pd.to_datetime(bursts['End time'], format='%Y-%m-%d/%H:%M', errors='coerce')
bursts['Duration (seconds)'] = (bursts['End time'] - bursts['Start time']).dt.total_seconds()

drift_rate = bursts["Frequency Change"] / bursts["Duration (seconds)"]
bursts["Drift Rate (Hz/s)"] = drift_rate
bursts.head()

,Start time,End time,High frequency (Hz),Low frequency (Hz),GOES,Flare,"Notes (corona loop brightening, etc.)",Other notes,Frequency Change,Duration (seconds),Drift Rate (Hz/s)
0,2023-02-08 03:20:00,2023-02-08 03:40:00,6000000,200000,F,NaN,NaN,NaN,5800000,1200.0,4833.333333
1,2023-02-08 04:05:00,2023-02-08 04:25:00,2000000,400000,F,NaN,NaN,NaN,1600000,1200.0,1333.333333
2,2023-02-09 07:15:00,2023-02-09 07:30:00,20000000,300000,T,F,NaN,SDO not shown,19700000,900.0,21888.888889
3,2023-02-10 02:56:00,2023-02-10 03:05:00,20000000,500000,F,NaN,NaN,NaN,19500000,540.0,36111.111111
4,2023-02-11 00:50:00,2023-02-11 01:05:00,10000000,600000,F,NaN,NaN,NaN,9400000,900.0,10444.444444


In [5]:
without_flare = bursts[(bursts["GOES"] == "T") & (bursts["Flare"] == "F")]
with_flare = bursts[(bursts["GOES"] == "T") & (bursts["Flare"] == "T")]
spot = bursts[bursts["Notes (corona loop brightening, etc.)"] == "Spot"]
loop = bursts[bursts["Notes (corona loop brightening, etc.)"] == "Loop"]
loop_spot = bursts[bursts["Notes (corona loop brightening, etc.)"] == "Loop & Spot"]

In [7]:
# print length
print("All bursts length: ", bursts.shape[0])
print("Without flare length: ", without_flare.shape[0])
print("With flare length: ", with_flare.shape[0])
print("All GOES correspondence: ", without_flare.shape[0] + with_flare.shape[0])
print("All spot bursts: ", spot.shape[0])
print("All loop bursts: ", loop.shape[0])
print("All loop and spot bursts: ", loop_spot.shape[0])

All bursts length:  185
Without flare length:  16
With flare length:  34
All GOES correspondence:  50
All spot bursts:  18
All loop bursts:  15
All loop and spot bursts:  2


In [9]:
# generate pop-up plot
from pytplot import get_data
from matplotlib import pyplot as plt
%matplotlib tk
from matplotlib.colors import LogNorm

In [11]:
# duration hist line plot function
def plot_duration_hist(data, name):
    duration = data["Duration (seconds)"].dropna()
    log_bins = np.logspace(np.log10(duration.min()), np.log10(duration.max()), 20)
    
    plt.figure()
    H, bins, patches = plt.hist(duration, bins=log_bins, edgecolor='black')
    
    plt.xscale('log')
    plt.xlim([duration.min() * 0.8, duration.max() * 1.2])  # Add padding
    plt.title(name + ' Duration Histogram')
    plt.xlabel('Duration (s)')
    plt.ylabel('Count')
    
    counts, bins = np.histogram(duration, bins=log_bins)
    plt.plot((bins[:-1] + bins[1:]) / 2, counts, label='Counts', color='red')
    
    plt.errorbar((bins[:-1] + bins[1:]) / 2, counts, yerr=np.sqrt(counts), fmt='o', color='red')
    
    plt.grid(True)
    plt.legend()
    plt.show()

plot_duration_hist(with_flare, "With Flare")
plot_duration_hist(without_flare, "Without Flare")
plot_duration_hist(spot, "Spot Burst")
plot_duration_hist(loop, "Loop Burst")
plot_duration_hist(loop_spot, "Loop and Spot Burst")

In [13]:
# plot line histogram all in 1 graph
def plot_combined_hist_line(datasets, labels):

    plt.figure(figsize=(10, 6))

    for data, label in zip(datasets, labels):
        duration = data["Duration (seconds)"].dropna()
        log_bins = np.logspace(np.log10(duration.min()), np.log10(duration.max()), 20)
        
        counts, bins = np.histogram(duration, bins=log_bins)
        plt.errorbar((bins[:-1] + bins[1:]) / 2, counts, yerr=np.sqrt(counts), fmt='o', label=label, linestyle='-')

    plt.xscale('log')
    plt.xlabel('Duration (s)')
    plt.ylabel('Count')
    plt.title('Combined Duration Histogram Lines')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_combined_hist_line(
    [with_flare, without_flare, spot, loop, loop_spot], 
    ["With Flare", "Without Flare", "Spot Burst", "Loop Burst", "Loop and Spot Burst"]
)

In [15]:
# freq change hist line plot function
def plot_duration_hist(data, name):
    freq_change = data["Frequency Change"].dropna()
    log_bins = np.logspace(np.log10(freq_change.min()), np.log10(freq_change.max()), 20)
    
    plt.figure()
    H, bins, patches = plt.hist(freq_change, bins=log_bins, edgecolor='black')
    
    plt.xscale('log')
    plt.xlim([freq_change.min() * 0.8, freq_change.max() * 1.2])  # Add padding
    plt.title(name + ' Frequency Change Histogram')
    plt.xlabel('Freq Change (Hz)')
    plt.ylabel('Count')
    
    counts, bins = np.histogram(freq_change, bins=log_bins)
    plt.plot((bins[:-1] + bins[1:]) / 2, counts, label='Counts', color='red')
    
    plt.errorbar((bins[:-1] + bins[1:]) / 2, counts, yerr=np.sqrt(counts), fmt='o', color='red')
    
    plt.grid(True)
    plt.legend()
    plt.show()

plot_duration_hist(with_flare, "With Flare")
plot_duration_hist(without_flare, "Without Flare")
plot_duration_hist(spot, "Spot Burst")
plot_duration_hist(loop, "Loop Burst")
plot_duration_hist(loop_spot, "Loop and Spot Burst")

In [17]:
# drift rate hist line plot function
def plot_duration_hist(data, name):
    drift_rate = data["Drift Rate (Hz/s)"].dropna()
    log_bins = np.logspace(np.log10(drift_rate.min()), np.log10(drift_rate.max()), 20)
    
    plt.figure()
    H, bins, patches = plt.hist(drift_rate, bins=log_bins, edgecolor='black')
    
    plt.xscale('log')
    plt.xlim([drift_rate.min() * 0.8, drift_rate.max() * 1.2])  # Add padding
    plt.title(name + ' Drift Rate Histogram')
    plt.xlabel('Drift Rate (Hz/s)')
    plt.ylabel('Count')
    
    counts, bins = np.histogram(drift_rate, bins=log_bins)
    plt.plot((bins[:-1] + bins[1:]) / 2, counts, label='Counts', color='red')
    
    plt.errorbar((bins[:-1] + bins[1:]) / 2, counts, yerr=np.sqrt(counts), fmt='o', color='red')
    
    plt.grid(True)
    plt.legend()
    plt.show()

plot_duration_hist(with_flare, "With Flare")
plot_duration_hist(without_flare, "Without Flare")
plot_duration_hist(spot, "Spot Burst")
plot_duration_hist(loop, "Loop Burst")
plot_duration_hist(loop_spot, "Loop and Spot Burst")

In [19]:
# poisson noise for with flare
means = with_flare[['Frequency Change', 'Duration (seconds)', 'Drift Rate (Hz/s)']].mean()
poisson_noise = np.sqrt(means)

print("\nPoisson Noise for with flare:\n", poisson_noise)

# poisson noise for without flare
means = without_flare[['Frequency Change', 'Duration (seconds)', 'Drift Rate (Hz/s)']].mean()
poisson_noise = np.sqrt(means)

print("\nPoisson Noise for without flare:\n", poisson_noise)

# poisson noise for with spot
means = spot[['Frequency Change', 'Duration (seconds)', 'Drift Rate (Hz/s)']].mean()
poisson_noise = np.sqrt(means)

print("\nPoisson Noise for with spot:\n", poisson_noise)

# poisson noise for with loop
means = loop[['Frequency Change', 'Duration (seconds)', 'Drift Rate (Hz/s)']].mean()
poisson_noise = np.sqrt(means)

print("\nPoisson Noise for with loop:\n", poisson_noise)

# poisson noise for with loop & spot
means = loop_spot[['Frequency Change', 'Duration (seconds)', 'Drift Rate (Hz/s)']].mean()
poisson_noise = np.sqrt(means)

print("\nPoisson Noise for with loop:\n", poisson_noise)


Poisson Noise for with flare:
 Frequency Change      3092.876065
Duration (seconds)      41.586197
Drift Rate (Hz/s)       89.479030
dtype: float64

Poisson Noise for without flare:
 Frequency Change      3516.834514
Duration (seconds)      58.159264
Drift Rate (Hz/s)       84.606788
dtype: float64

Poisson Noise for with spot:
 Frequency Change      2798.213716
Duration (seconds)      43.931765
Drift Rate (Hz/s)       69.614956
dtype: float64

Poisson Noise for with loop:
 Frequency Change      3423.448554
Duration (seconds)      37.469988
Drift Rate (Hz/s)      109.312637
dtype: float64

Poisson Noise for with loop:
 Frequency Change      3761.648575
Duration (seconds)      45.497253
Drift Rate (Hz/s)      107.513184
dtype: float64
